# 2-3. Retriever 패턴: similarity / MMR / threshold / 메타필터 / 멀티테넌트

## 학습 목표
- similarity / MMR / threshold 기반 Retriever 의 동작 차이를 구분하고 상황에 맞게 선택한다.
- 메타데이터(소스, 작성일, 부서 등)를 활용한 필터링을 `$and` / `$in` 등으로 구성할 수 있다.
- 금융권 **멀티테넌트 격리**를 `tenant_id` 필터로 구현하고, 콜렉션 분리 vs 동일 콜렉션 + 필터의 트레이드오프를 설명한다.
- Chroma 의 기본 L2 공간 대신 cosine 공간으로 색인해야 `similarity_search_with_relevance_scores` 가 0~1 범위로 동작하는 이유를 이해한다.

## 핵심 키워드
`similarity` `MMR` `score_threshold` `metadata filter` `multi-tenant` `tenant_id` `as_retriever` `hnsw:space=cosine`

> 🔒 모든 리트리버는 로컬 Chroma 에 구축된 인덱스에서 동작하여 폐쇄망 친화적이다.

In [1]:
import os
os.environ['ANONYMIZED_TELEMETRY'] = 'False'

import sys; sys.path.insert(0, '../..')
from common import get_chat_model, get_embeddings, provider_badge
print(provider_badge())

☁️ OpenAI | model=openai/gpt-4o-mini


In [2]:
# 실습용 코퍼스: 금융 도메인의 다중 테넌트 문서를 모상
from datetime import datetime, timedelta
from langchain_core.documents import Document

now = datetime(2026, 4, 17)
D = lambda d, days=0, **m: Document(page_content=d, metadata={'date': (now - timedelta(days=days)).date().isoformat(), **m})

docs = [
    # tenant A — 리테일 은행 (B2C)
    D('결제카드 분실시 고객센터 1588-0000으로 즉시 신고하여야 합니다.', days=10, tenant_id='retail_bank', source='cs_manual', topic='lost_card'),
    D('청약철회는 계약일로부터 14일 이내에 서면으로 가능합니다.', days=120, tenant_id='retail_bank', source='terms', topic='withdrawal'),
    D('2026년 1월부터 오픈뱅킹 수수료가 무료로 전환됩니다.', days=90, tenant_id='retail_bank', source='announce', topic='fee'),
    # tenant B — 투자증권 (IB)
    D('기관투자자 전용 프라임 브로커리지 수수료는 거래액의 0.03%입니다.', days=5, tenant_id='securities', source='ib_terms', topic='fee'),
    D('ELS 상품의 중도상환 수수료는 잔존기간에 따라 차등 적용됩니다.', days=60, tenant_id='securities', source='product', topic='product'),
    # tenant C — 보험
    D('자동차보험 신규 가입은 면허 보유 1년 경과 시 우대 적용 가능.', days=15, tenant_id='insurance', source='policy', topic='auto'),
    D('실속 의료보험은 보장 제한 사항을 약관 제5조에서 확인하세요.', days=400, tenant_id='insurance', source='terms', topic='medical'),
]
print(f'총 {len(docs)}개 문서, 테넌트 종류: {sorted({d.metadata["tenant_id"] for d in docs})}')

총 7개 문서, 테넌트 종류: ['insurance', 'retail_bank', 'securities']


In [ ]:
import shutil
from pathlib import Path
from langchain_community.vectorstores import Chroma

CHROMA_DIR = Path('./_store/chroma_retriever')
if CHROMA_DIR.exists():
    shutil.rmtree(CHROMA_DIR)
CHROMA_DIR.parent.mkdir(parents=True, exist_ok=True)

vs = Chroma.from_documents(
    docs,
    embedding=get_embeddings(),
    persist_directory=str(CHROMA_DIR),
    collection_name='fin_multi_tenant',
    collection_metadata={'hnsw:space': 'cosine'},
)
print('인덱싱된 문서:', vs._collection.count())

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


인덱싱된 문서: 7


## 1. similarity vs MMR vs threshold

리트리버 선택은 "가장 비슷한 문서 k개 가져오기"만의 문제가 아니다. 실제 금융 상담 챗봇에서는
- **중복되지 않는 다양한 근거**를 모아 답해야 하거나,
- **근거가 부족할 때는 차라리 '모릅니다'라고 답해야** 할 때가 많다.

세 모드 모두 `vs.as_retriever(search_type=..., search_kwargs=...)` 로 교체 가능하므로,
동일 질의에 대한 결과 차이를 직접 비교해 본다.

### 1.1 similarity (기본 모드)
- **무엇**: 질의 임베딩과 코사인 유사도가 가장 높은 상위 k개를 그대로 반환.
- **언제 쓰나**: 질의에 대한 정답 문서가 1개로 명확한 경우 (FAQ, 고객센터 안내번호 조회 등).
- **주의**: 같은 내용을 여러 버전으로 기술한 약관/공지가 있으면 k개가 전부 중복일 수 있다. → 답변이 한쪽으로 치우침.
- **금융 예시**: "카드 분실 신고 번호?" 처럼 정답이 한 문서에 수렴하는 질의에 적합.

In [4]:
q = '카드 분실 신고 방법 알려줘'

sim_retriever = vs.as_retriever(search_type='similarity', search_kwargs={'k': 3})
for d in sim_retriever.invoke(q):
    print(f'[similarity] {d.metadata["tenant_id"]}/{d.metadata["topic"]}: {d.page_content[:50]}…')

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[similarity] retail_bank/lost_card: 결제카드 분실시 고객센터 1588-0000으로 즉시 신고하여야 합니다.…
[similarity] insurance/medical: 실속 의료보험은 보장 제한 사항을 약관 제5조에서 확인하세요.…
[similarity] retail_bank/withdrawal: 청약철회는 계약일로부터 14일 이내에 서면으로 가능합니다.…


### 1.2 MMR (Maximal Marginal Relevance)
- **무엇**: `관련성 - λ × 이미 선택된 문서와의 유사도` 를 최대화하도록 k개를 **순차적으로** 뽑는다. `fetch_k` 개의 후보를 먼저 모은 뒤 그 안에서 다양성을 고려해 고른다.
- **왜 필요한가**: 약관·공지·FAQ 등 **같은 주제를 여러 버전으로 기술한 문서**가 많은 금융 도메인에서 similarity 는 같은 말을 여러 번 가져오기 쉽다. MMR 은 의미 중복을 줄여 LLM 이 더 폭넓은 근거로 답하도록 만든다.
- **파라미터 감각**
  - `fetch_k` : 후보 풀 크기. 보통 `k × 3~5`. 너무 작으면 다양성이 사라짐.
  - `lambda_mult` : 1 에 가까울수록 순수 유사도(= similarity 에 수렴), 0 에 가까울수록 다양성 우선. 실전 기본값 0.3~0.5.
- **언제 쓰면 안 되나**: 질의에 대한 정답 문서가 **딱 하나** 있어야 하는 경우(예: 특정 상품의 수수료율). 다양성을 강제하면 엉뚱한 문서가 끼어든다.
- **금융 예시**: "청약철회 관련해서 알려줘" → 약관 조항 + CS 매뉴얼 + 공지사항을 골고루 조합해야 안내 품질이 올라간다.

In [5]:
# MMR 실행: 동일 질의에 대해 similarity 와 결과가 어떻게 달라지는지 비교한다.
# fetch_k  : 후보 풀의 크기 (최종 k 개를 뽑기 위해 먼저 넉넉히 가져옴)
# lambda_mult : 1 에 가까울수록 엄격한 유사도, 0 에 가까울수록 다양성 우선
mmr_retriever = vs.as_retriever(search_type='mmr', search_kwargs={'k': 3, 'fetch_k': 7, 'lambda_mult': 0.3})
for d in mmr_retriever.invoke(q):
    print(f'[mmr]        {d.metadata["tenant_id"]}/{d.metadata["topic"]}: {d.page_content[:50]}…')

[mmr]        retail_bank/lost_card: 결제카드 분실시 고객센터 1588-0000으로 즉시 신고하여야 합니다.…
[mmr]        insurance/medical: 실속 의료보험은 보장 제한 사항을 약관 제5조에서 확인하세요.…
[mmr]        securities/fee: 기관투자자 전용 프라임 브로커리지 수수료는 거래액의 0.03%입니다.…


### 1.3 score_threshold (유사도 임계값)
- **무엇**: 유사도가 `score_threshold` 이상인 문서만 반환. k 개를 채우지 못하면 그대로 0~k 개만 돌려준다.
- **왜 필요한가**: RAG 의 가장 흔한 실패 모드는 **"근거가 없는데도 LLM 이 그럴듯하게 지어내는 것"**(hallucination). 임계값으로 잘라내면 "근거가 없을 때" 를 명시적으로 감지할 수 있고, 프롬프트에서 "근거 문서가 비어 있으면 '모릅니다' 라고 답하라" 로 연결할 수 있다.
- **점수 해석(cosine 공간 기준)**
  - `0.9 ~ 1.0` : 거의 동일한 문서. 중복 색인 의심.
  - `0.6 ~ 0.9` : 같은 주제의 관련 문서.
  - `0.3 ~ 0.6` : 어렴풋이 관련. 노이즈가 섞이기 쉬움.
  - `≤ 0.3` : 거의 무관.
- **임계값 정하는 법**: 보통 **정답/오답 라벨이 붙은 평가셋**(다음 노트북 `05_eval_ragas`)에서 recall 과 precision 을 함께 보며 튜닝한다. 규정 해석처럼 오답 리스크가 큰 도메인은 임계값을 보수적으로(높게) 잡는다.
- **주의**: 앞선 셀에서 `hnsw:space='cosine'` 을 지정하지 않으면 점수가 음수로 나오거나 경고가 뜬다. 반드시 cosine 공간으로 색인한 뒤 임계값을 논하자.

In [6]:
# score threshold — 유사도가 일정 이상인 결과만 반환 (hallucination 방지에 유용)
thr_retriever = vs.as_retriever(search_type='similarity_score_threshold', search_kwargs={'k': 5, 'score_threshold': 0.4})
hits = thr_retriever.invoke(q)
print(f'[threshold=0.4] hits={len(hits)}')
for d in hits:
    print(f'  {d.metadata["tenant_id"]}/{d.metadata["topic"]}: {d.page_content[:50]}…')

# 임계값을 높이면 결과가 0개가 될 수 있다 → "못찾았습니다" 처리로 연결
thr2 = vs.as_retriever(search_type='similarity_score_threshold', search_kwargs={'k': 5, 'score_threshold': 0.85})
print(f'[threshold=0.85] hits={len(thr2.invoke(q))}  ← 텍스트 없으면 LLM이 "모르겠습니다" 로 답하도록 설정')

[threshold=0.4] hits=1
  retail_bank/lost_card: 결제카드 분실시 고객센터 1588-0000으로 즉시 신고하여야 합니다.…


No relevant docs were retrieved using the relevance score threshold 0.85


[threshold=0.85] hits=0  ← 텍스트 없으면 LLM이 "모르겠습니다" 로 답하도록 설정


## 2. 메타데이터 필터링

**"검색 전에 검색 공간을 좁히는"** 기법. 벡터 유사도만으로는 "최근 30일 공지만", "은행 사업부 문서만", "보안등급 1등급 이하만" 같은 요건을 충족할 수 없다. 문서를 색인할 때 메타데이터(`source`, `date`, `tenant_id`, `topic` 등)를 함께 넣어두면 질의 시점에 **하드 필터**를 걸 수 있다.

### Chroma 필터 연산자 요약
| 연산자 | 의미 | 예시 |
|---|---|---|
| `$eq` / `$ne` | 같음 / 다름 | `{'source': {'$eq': 'terms'}}` |
| `$in` / `$nin` | 값 포함 / 미포함 | `{'topic': {'$in': ['fee', 'withdrawal']}}` |
| `$gt` / `$gte` / `$lt` / `$lte` | 대소 비교 (숫자/비교 가능한 값) | `{'priority': {'$gte': 3}}` |
| `$and` / `$or` | 논리 결합 | `{'$and': [{'source': {'$eq': 'terms'}}, {'topic': {'$eq': 'fee'}}]}` |

> ⚠️ Chroma 의 `$gte` / `$lte` 는 **숫자형에만** 적용된다. 문자열로 저장된 `date` 에 범위 조건을 주려면 (1) 색인할 때 epoch(int) 로 함께 저장하거나, (2) `similarity_search` 로 넓게 뽑은 뒤 Python 에서 후처리한다.

### 언제 쓰나
- **RBAC(Role-Based Access Control)**: 직원 권한에 따라 문서군을 제한. 아래 3번 멀티테넌트 패턴의 토대.
- **시간 기반 룰**: "최근 개정된 약관만 인용" 같은 규제 요건.
- **주제/출처 구분**: 공지 vs 약관 vs 내부 매뉴얼을 한 컬렉션에 함께 보관하면서도 질의마다 다른 출처를 참조하고 싶을 때.

In [7]:
# source == 'terms' 인 문서만 필터링
# Chroma 의 $gte 는 숫자형에만 적용되므로, 문자열 date 는 $eq 또는 Python 후처리로 다룬다.
recent_terms = vs.as_retriever(search_kwargs={
    'k': 5,
    'filter': {'source': {'$eq': 'terms'}},
})
for d in recent_terms.invoke('약관 관련 조항'):
    print(f'  date={d.metadata["date"]} source={d.metadata["source"]}: {d.page_content[:50]}…')

  date=2025-03-13 source=terms: 실속 의료보험은 보장 제한 사항을 약관 제5조에서 확인하세요.…
  date=2025-12-18 source=terms: 청약철회는 계약일로부터 14일 이내에 서면으로 가능합니다.…


In [8]:
# $and / $in 조합 — source 가 'terms' 이면서 topic 이 fee 또는 withdrawal 인 것만
combo = vs.as_retriever(search_kwargs={
    'k': 5,
    'filter': {
        '$and': [
            {'source': {'$eq': 'terms'}},
            {'topic': {'$in': ['fee', 'withdrawal']}},
        ]
    },
})
for d in combo.invoke('수수료 또는 청약철회 관련 조항'):
    print(f'  {d.metadata["source"]}/{d.metadata["topic"]}: {d.page_content[:60]}…')

  terms/withdrawal: 청약철회는 계약일로부터 14일 이내에 서면으로 가능합니다.…


## 3. 멀티테넌트 격리 패턴

### 왜 금융권에서 필수인가
하나의 그룹사 안에도 리테일 은행·투자증권·보험처럼 **고객 정보와 상품 정보가 법적으로 분리**되어야 하는 사업부가 공존한다. 예:
- 리테일 은행 직원이 투자증권의 기관 고객 약관을 열람하면 **정보장벽(Chinese Wall) 위반**.
- 보험 약관을 은행 CS 답변에 인용하면 **불완전판매** 리스크.

따라서 "같은 벡터 DB에 전사 문서를 넣되 각 리트리버는 내 테넌트만 보이도록 필터로 격리한다" 는 패턴이 표준이다.

### 두 가지 격리 방식 비교
| 방식 | 장점 | 단점 | 언제 선택 |
|---|---|---|---|
| **테넌트별 콜렉션 분리** (`collection_name='retail_bank'` 등) | 물리적 격리, 권한 관리 심플 | 공통 지식(공용 FAQ 등) 중복 색인 필요, 크로스 질의 불가 | 규제 요건이 강할 때 (권장) |
| **동일 콜렉션 + `tenant_id` 필터** (아래 예시) | 저장 공간 효율, 공통 지식 재활용 | 필터를 **빠뜨리는 실수**가 곧 정보유출 | 테넌트 수가 많고 공용 자료가 많을 때 |

### 구현 시 반드시 지킬 것
1. **세션 진입 시점에 `tenant_id` 를 확정**하고 그 값을 하드코딩한 retriever 객체만 노출. 애플리케이션 코드에서 매번 필터를 손으로 작성하게 두면 어디선가 빠뜨린다.
2. **감사 로그**: 사용자 `tenant_id`, 실제 필터 조건, 반환 문서 ID 를 함께 기록. 규제 감사 시 "정말 격리되었는가?" 를 역추적 가능해야 한다.
3. **교차 노출 테스트**: 테넌트 A 전용 retriever 로 테넌트 B 질의를 던져 **0건이 보장되는지** CI 단계에서 자동 검증.

In [9]:
def tenant_retriever(tenant_id: str, k: int = 3):
    """세션 시작 시 테넌트를 고정한 retriever 를 만들어 반환한다.
    필터를 객체에 박아두면 호출부에서 실수로 필터를 빠뜨릴 여지가 없다."""
    return vs.as_retriever(
        search_type='similarity',
        search_kwargs={'k': k, 'filter': {'tenant_id': {'$eq': tenant_id}}},
    )

retail = tenant_retriever('retail_bank')
sec    = tenant_retriever('securities')

q2 = '수수료 구조 알려줘'
print('--- retail_bank ---')
for d in retail.invoke(q2):
    print(f'  {d.metadata["tenant_id"]}/{d.metadata["source"]}: {d.page_content[:60]}')
print('--- securities ---')
for d in sec.invoke(q2):
    print(f'  {d.metadata["tenant_id"]}/{d.metadata["source"]}: {d.page_content[:60]}')

--- retail_bank ---
  retail_bank/announce: 2026년 1월부터 오픈뱅킹 수수료가 무료로 전환됩니다.
  retail_bank/cs_manual: 결제카드 분실시 고객센터 1588-0000으로 즉시 신고하여야 합니다.
  retail_bank/terms: 청약철회는 계약일로부터 14일 이내에 서면으로 가능합니다.
--- securities ---
  securities/product: ELS 상품의 중도상환 수수료는 잔존기간에 따라 차등 적용됩니다.
  securities/ib_terms: 기관투자자 전용 프라임 브로커리지 수수료는 거래액의 0.03%입니다.


In [ ]:
# 교차 노출 테스트: retail 직원이 securities 문서를 조회할 수 있는가?
result = retail.invoke('ELS 중도상환 수수료')
only_retail = all(d.metadata['tenant_id'] == 'retail_bank' for d in result)
print(f'retail 테넌트만 반환되는가? {only_retail}')
for d in result:
    print(f'  tenant={d.metadata["tenant_id"]}  content={d.page_content[:40]}…')

retail 테넌트만 반환되는가? True
  tenant=retail_bank  content=2026년 1월부터 오픈뱅킹 수수료가 무료로 전환됩니다.…
  tenant=retail_bank  content=결제카드 분실시 고객센터 1588-0000으로 즉시 신고하여야 합니다.…
  tenant=retail_bank  content=청약철회는 계약일로부터 14일 이내에 서면으로 가능합니다.…


: 

### 실전 팁
- **콜렉션 분리**가 가능하다면 테넌트별 콜렉션 생성이 가장 안전하다 (인덱스·스토리지의 물리적 분리).
- 부득이 동일 콜렉션을 써야 한다면 Retriever 생성 시 필터를 하드코딩해 **우연히 필터를 빼는 실수**를 원천 차단한다.
- 감사 로그에는 사용자의 `tenant_id` 와 실제 적용된 필터 조건, 반환된 문서 ID 가 함께 남아야 한다.

## 정리

| 패턴 | 쓰는 상황 | 쓰면 안 되는 상황 |
|---|---|---|
| **similarity** | 정답 문서가 1개로 명확한 질의 (FAQ, 고정 안내 번호) | 같은 내용의 중복 문서가 많을 때 (답이 편향됨) |
| **MMR** | 약관/공지/FAQ 를 다각도로 조합해야 할 때 | 정답이 딱 하나인 질의 (엉뚱한 문서가 섞임) |
| **threshold** | "근거가 없으면 모른다고 답해야" 하는 규정 해석·컴플라이언스 | 반드시 k 개를 뽑아야 하는 UX (빈 결과가 허용 안 됨) |
| **metadata filter** | 부서/날짜/보안등급 등 사전 조건이 있을 때 | 조건이 동적이라 사전 색인 단계에서 메타를 못 넣는 경우 |
| **tenant filter** | 금융권 사업부별 정보 격리 (Chinese Wall) | 테넌트가 1개뿐이면 과잉 설계 |

## 더 읽어보기
- [LangChain Retrievers Concept](https://python.langchain.com/docs/concepts/retrievers/)
- [MMR 논문 (Carbonell & Goldstein, 1998)](https://dl.acm.org/doi/10.1145/290941.291025)
- [Chroma Collection Configuration (hnsw:space 등)](https://docs.trychroma.com/docs/collections/configure)